## Tool Calling
让模型调用你的函数

- 用户说："BOM-1001 变更影响什么？"
-      ↓
- 模型判断："这个问题需要查 BOM 数据，我有一个 get_bom_change 工具可以用"
-      ↓
- 模型返回："我要调用 get_bom_change，参数是 BOM-1001"
-      ↓
- 你的代码执行 get_bom_change("BOM-1001")，得到结果
-      ↓
- 把结果送回给模型
-      ↓
- 模型基于结果生成最终回答："BOM-1001 变更影响螺丝 M3 和钣金件 A12，涉及产线 SMT-2"

In [5]:
# 从 .env 读取api_key
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print("api_key加载成功" if api_key else "未找到api_key")

api_key加载成功


## 第 1 步：写一个工具函数

In [6]:
def get_bom_change(bom_id):
    # 根据 BOM 编号查询变更影响到的物料和产线
    
    # 模拟数据（实际项目中这里会查数据库或调 API）
    bom_data = {
        "BOM-1001": "影响物料：螺丝 M3, 钣金件 A12；涉及产线：SMT-2",
        "BOM-1002": "影响物料：电池 5000mAh, 外壳 B 款；涉及产线：组装-3",
        "BOM-1003": "影响物料：PCB 主板, 散热片；涉及产线：DIP-1",
    }
    return bom_data.get(bom_id, "未找到该 BOM 的变更记录")

print(get_bom_change("BOM-1001"))
print(get_bom_change("BOM-9999"))

影响物料：螺丝 M3, 钣金件 A12；涉及产线：SMT-2
未找到该 BOM 的变更记录


## 第 2 步：把工具函数“注册”给模型
"type":"function" 固定写法 <br>
function 
- name：函数名，必须和你 Python 函数名一致
- description：告诉模型这个函数是干什么的，写得越清楚，模型越知道什么时候该用它
- parameters：参数定义
- - "type":"object" 固定写法，表示参数整体是一个对象（字典），因为函数参数通常以键值对（Key-Value）的形式传递
- - properties
- - - bom_id：参数名
- - - - type：参数类型（string/number/array）
- - - - description：参数说明
- - required：哪些参数是必填的

In [7]:
tools = [
    {
        "type":"function",
        "function":{
            "name":"get_bom_change",
            "description":"根据 BOM 编号查询变更影响到的物料和产线",
            "parameters":{
                "type":"object",
                "properties":{
                    "bom_id":{
                        "type":"string",
                        "description":"BOM 编号，如 BOM-1001"
                    }
                    #假设存在多个参数
#                     ,
#                     "bom_id":{
#                         "type":"string",
#                         "description":"BOM 编号，如 BOM-1001"
#                     }
                },
                "required":["bom_id"]
            }
        }
    }
]

## 第 3 步：第一次调模型，带上 tools

In [9]:
import requests
import json

#  --- 配置 ---
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization":f"Bearer {api_key}",
    "Content-type":"application/json"
}

messages = [
    {"role":"system","content":"你是制造企业的 AI 助理，回答简洁专业，用中文。"},
    {"role":"user","content":"BOM-1001 变更会影响什么？"}
]

data = json.dumps({
    "model":"nvidia/nemotron-3.5-lightning:free",
    "messages":messages,
    "tools":tools,
    "tool_choice":"auto"
})

resp = requests.post(url=url, headers=headers, data=data)
resp = resp.json()
print(resp)

{'id': 'gen-1792042684-9vvLXUKCfhpEhjaUwAmo', 'object': 'chat.completion', 'created': 1792042684, 'model': 'nvidia/nemotron-3.5-lightning:free', 'provider': 'Nvidia', 'system_fingerprint': None, 'service_tier': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'tool_calls', 'native_finish_reason': 'tool_calls', 'message': {'role': 'assistant', 'content': None, 'refusal': None, 'reasoning': 'We need to answer user: "BOM-1001 变更会影响什么？" As AI assistant for manufacturing, concise professional Chinese. Use tool get_bom_change with bom_id "BOM-1001".', 'tool_calls': [{'type': 'function', 'index': 0, 'id': 'call-876ecdba-fbf4-42a0-932b-ba2cc8f305f0', 'function': {'name': 'get_bom_change', 'arguments': '{"bom_id":"BOM-1001"}'}}], 'reasoning_details': [{'type': 'reasoning.text', 'text': 'We need to answer user: "BOM-1001 变更会影响什么？" As AI assistant for manufacturing, concise professional Chinese. Use tool get_bom_change with bom_id "BOM-1001".', 'format': 'unknown', 'index': 0}]}}

### 上述resp.json所得结果如下：

- content 是 null —— 模型没有直接回答
- tool_calls 出现了 —— 模型决定要调工具
- function.name 是 get_bom_change —— 模型选择了正确的函数
- function.arguments 是 {"bom_id": "BOM-1001"} —— 模型正确提取了参数

## 第 4 步：解析 tool_calls 并执行函数
- 检查模型是否返回了 tool_calls
- 如果有，解析出函数名和参数
- 调用对应的 Python 函数
- 把结果送回给模型

In [16]:
# 从响应中取出消息
msg = resp["choices"][0]["message"]
# 把模型的消息加入对话历史
messages.append(msg)

# 检查是否有工具调用
# msg["tool_calls"] 和 msg.get("tool_calls") 的区别在于：‌前者在键不存在时会直接报错，后者会返回 None 而不是报错
if(msg.get("tool_calls")):
    for tc in msg["tool_calls"]:
        function_name = tc["function"]["name"]
        # tc["function"]["arguments"]：取出来的是‌字符串‌，里面是 JSON 格式的参数，比如 '{"city": "北京"}'。
        # json.loads(tc["function"]["arguments"])：把这个 JSON ‌字符串解析成 Python 字典‌，之后才能用 args["city"] 这种形式取参数。
        # 简单说，json.loads() 是必需的转换步骤，因为 arguments 本身是字符串，不解析没法直接用
        function_args = json.loads(tc["function"]["arguments"])
        
        print(f"模型决定调用函数：{function_name}")
        print(f"参数：{function_args}")
        
        # 根据函数名执行对应的 Python 函数
        if function_name == "get_bom_change":
            function_result = get_bom_change(function_args["bom_id"])
            print(f"函数返回：{function_result}")
        
            # 把函数执行结果送回给模型
            messages.append({
                "role":"tool",
                "tool_call_id":tc["id"], # 告诉模型这是对应哪次调用
                "content":function_result # 函数返回的结果
            })
    
    # 第二次调模型，带着工具执行结果
    data2 = json.dumps({
        "model":"nvidia/nemotron-3.5-lightning:free",
        "messages":messages,
        "tools":tools,
        "tool_choice":"auto"
    })
    
    resp2 = requests.post(url=url, headers=headers, data=data2)
    resp2 = resp2.json()
    resp2 = resp2["choices"][0]["message"]["content"]
    
    print("\n=== 最终回答 ===")
    print(resp2)

else:
    # 模型直接回答了，没有调工具
    print("模型直接回答：",msg["content"])

模型决定调用函数：get_bom_change
参数：{'bom_id': 'BOM-1001'}
函数返回：影响物料：螺丝 M3, 钣金件 A12；涉及产线：SMT-2

=== 最终回答 ===
BOM‑1001 变更将影响的物料及产线如下：

- **受影响物料**：螺丝 M3、钣金件 A12  
- **涉及产线**：SMT‑2  

请相应更新物料清单，并对 SMT‑2 产线进行检查与调配。


## 第 5 步：封装成通用函数
把上面的逻辑封装成一个可复用的函数，以后加新工具只需要扩展 available_functions 字典

In [28]:
import requests
import json

#  -- 配置 --
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization":f"Bearer {api_key}",
    "Content-Type":"application/json",
}

tools = [
    {
        "type":"function",
        "function":{
            "name":"get_bom_change",
            "description":"根据 BOM 编号查询变更影响到的物料和产线",
            "parameters":{
                "type":"object",
                "properties":{
                    "bom_id":{
                        "type":"string",
                        "description":"BOM 编号，如 BOM-1001"
                    }
                },
                "required":["bom_id"]
            }
        }
    }
]

available_functions = {
    "get_bom_change":get_bom_change
}

def run_agent(user_input, tools, available_functions):
    """
    通用 Agent 函数
    
    参数：
        user_input: string, 用户输入
        tools: list, 工具定义列表
        available_functions: dict, 函数名到实际函数的映射
    
    返回：
        string, 最终回答
    """
    # 第一次调模型
    data = json.dumps({
        "model":"nvidia/nemotron-3.5-lightning:free",
        "messages":[
            {"role":"system", "content":"你是制造企业的 AI 助理，回答简洁专业，用中文。"},
            {"role":"user", "content": user_input}
        ],
        "tools":tools,
        "tool_choice":"auto"
    })
    
    resp = requests.post(url=url, headers=headers, data=data)
    resp = resp.json()["choices"][0]["message"]
    
    messages.append(resp)
    
    # 处理工具调用
    if(resp.get("tool_calls")):
        for tc in resp["tool_calls"]:
            function_name = tc["function"]["name"]
            function_args = json.loads(tc["function"]["arguments"])
            print(f"模型决定调用函数：{function_name}")
            print(f"参数：{function_args}")
            
            # 从 available_functions 中找到对应的函数并执行
            if function_name in available_functions:
                func_to_call = available_functions[function_name]
                # 加上 **：Python 解释器会将字典中的每个键值对拆解，转换为 key=value 的形式传入
                function_result = func_to_call(**function_args)
                print(f"函数返回：{function_result}")
                
                messages.append({
                    "role":"tool",
                    "tool_call_id":tc["id"],
                    "content": function_result
                })
            
        # 第二次调模型
        data2 = json.dumps({
            "model":"nvidia/nemotron-3.5-lightning:free",
            "messages":messages,
            "tools":tools,
            "tool_choice":"auto"
        })

        resp2 = requests.post(url=url, headers=headers, data=data2)
        resp2 = resp2.json()["choices"][0]["message"]
            
        return resp2["content"]
    
    else:
        
        return resp["content"]

# 测试
print("agent回答：",run_agent("BOM-1002 变更有什么影响？", tools, available_functions))
print("---")
print("agent回答：",run_agent("帮我查一下 BOM-1003", tools, available_functions))
print("---")
print("agent回答：",run_agent("今天天气怎么样？", tools, available_functions))  # 模型发现没有天气工具，会直接回答不知道

模型决定调用函数：get_bom_change
参数：{'bom_id': 'BOM-1002'}
函数返回：影响物料：电池 5000mAh, 外壳 B 款；涉及产线：组装-3
agent回答： 根据BOM‑1002变更，影响物料：电池 5000 mAh、外壳 B 款；涉及产线：组装‑3。
---
模型决定调用函数：get_bom_change
参数：{'bom_id': 'BOM-1003'}
函数返回：影响物料：PCB 主板, 散热片；涉及产线：DIP-1
agent回答： BOM-1003 变更影响的物料和产线：

- **影响物料**：PCB 主板、散热片  
- **涉及产线**：DIP-1  

如果需要进一步分析或跟进变更流程，请告知。
---
agent回答： 作为制造企业的AI助手，我专注于生产管理与BOM变更等业务问题，不提供实时天气信息。请您查询当地天气应用获取最新情况。


## 第 6 步：加第二个工具，让模型自己做选择

In [29]:
def get_material_stock(material_name):
    """
    查询物料库存
    
    参数：
        material_name: string, 物料名称
    
    返回：
        string, 库存信息
    """
    
    stock_data = {
        "螺丝 M3": "库存充足，现有 5000 个",
        "钣金件 A12": "库存紧张，仅剩 120 个，建议补货",
        "电池 5000mAh": "库存充足，现有 2000 个",
        "外壳 B 款": "无库存，已停产",
    }
    
    return stock_data.get(material_name, f"未找到物料{material_name}的库存信息")

### 注册第二个工具

In [31]:
tools_v2 = [
    {
        "type":"function",
        "function":{
            "name":"get_bom_change",
            "description":"根据 BOM 编号查询变更影响到的物料和产线",
            "parameters":{
                "type":"object",
                "properties":{
                    "bom_id":{
                        "type":"string",
                        "description":"BOM 编号，如 BOM-1001"
                    }
                },
                "required":["bom_id"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"get_material_stock",
            "description":"查询某个物料的当前库存情况",
            "parameters":{
                "type":"object",
                "properties":{
                    "material_name":{
                        "type":"string",
                        "description":"物料名称，如 螺丝 M3"
                    }
                },
                "required":["material_name"]
            }
        }
    }
]

available_functions_v2 = {
    "get_bom_change": get_bom_change,
    "get_material_stock": get_material_stock
}

In [34]:
print("agent回答：",run_agent("BOM-1001 变更影响什么？", tools_v2, available_functions_v2))
print("---")
print("agent回答：",run_agent("螺丝 M3 还有多少库存？", tools_v2, available_functions_v2))
print("---")
print("agent回答：",run_agent("BOM-1002 变更后，受影响的物料库存够吗？", tools_v2, available_functions_v2))

模型决定调用函数：get_bom_change
参数：{'bom_id': 'BOM-1001'}
函数返回：影响物料：螺丝 M3, 钣金件 A12；涉及产线：SMT-2
agent回答： BOM‑1001 变更影响的物料包括：螺丝 M3、钣金件 A12；涉及的产线为 SMT‑2。
---
模型决定调用函数：get_material_stock
参数：{'material_name': '螺丝 M3'}
函数返回：库存充足，现有 5000 个
agent回答： 螺丝 M3的当前库存为**5000个**，库存充足。
---
模型决定调用函数：get_bom_change
参数：{'bom_id': 'BOM-1002'}
函数返回：影响物料：电池 5000mAh, 外壳 B 款；涉及产线：组装-3
agent回答： None


单轮 Tool Calling的局限性： <br>
最后一个问题特别有意思——模型可能需要先调 get_bom_change 查到影响物料，再调 get_material_stock 查库存。但目前我们的 run_agent 只支持一轮工具调用，之后我们会升级成多轮循环。